In [ ]:
# Import SparkSession to initialize Spark.
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('Practice_rdd').master('local[*]').getOrCreate()
sc = spark.sparkContext
print(sc)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 18:53:33 WARN Utils: Your hostname, TechBuddy, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/02 18:53:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 18:53:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=Practice_rdd>


In [4]:
li = [i for i in range(1,11)]
rdd = sc.parallelize(li)
rdd.getNumPartitions()
rdd.glom().collect()

[[], [1], [], [2], [3], [], [4], [5], [], [6], [], [7], [8], [], [9], [10]]

In [6]:
import csv

file_path = "/root/pyspark/real_data_practice/sales.csv"

lines = sc.textFile(file_path)
header = lines.first()

# Parse each CSV line into a list of values, excluding the header.
rows_rdd = (
    lines
    .filter(lambda line: line != header)
    .map(lambda line: next(csv.reader([line])))
)

rows_rdd.take(6)

[['1001', 'C101', 'Bangalore', 'Laptop', '80000'],
 ['1002', 'C102', 'Delhi', 'Phone', '30000'],
 ['1003', 'C101', 'Bangalore', 'Mouse', '2000'],
 ['1004', 'C103', 'Mumbai', 'Laptop', '75000'],
 ['1005', 'C104', 'Delhi', 'Keyboard', '5000'],
 ['1006', 'C101', 'Bangalore', 'Phone', '25000']]

In [11]:
sales = rows_rdd.map(lambda x: tuple((', '.join(x)).split(', ')))
sales.collect()

[('1001', 'C101', 'Bangalore', 'Laptop', '80000'),
 ('1002', 'C102', 'Delhi', 'Phone', '30000'),
 ('1003', 'C101', 'Bangalore', 'Mouse', '2000'),
 ('1004', 'C103', 'Mumbai', 'Laptop', '75000'),
 ('1005', 'C104', 'Delhi', 'Keyboard', '5000'),
 ('1006', 'C101', 'Bangalore', 'Phone', '25000')]

In [13]:
sales = rows_rdd.map(lambda x: (x[0],x[1],x[3]))
sales.collect()

[('1001', 'C101', 'Laptop'),
 ('1002', 'C102', 'Phone'),
 ('1003', 'C101', 'Mouse'),
 ('1004', 'C103', 'Laptop'),
 ('1005', 'C104', 'Keyboard'),
 ('1006', 'C101', 'Phone')]

In [15]:
rows_rdd.map(lambda x:float(x[4]) > 5000).collect()

[True, True, False, True, False, True]

In [16]:
rows_rdd.filter(lambda x:float(x[4]) > 5000).collect()

[['1001', 'C101', 'Bangalore', 'Laptop', '80000'],
 ['1002', 'C102', 'Delhi', 'Phone', '30000'],
 ['1004', 'C103', 'Mumbai', 'Laptop', '75000'],
 ['1006', 'C101', 'Bangalore', 'Phone', '25000']]

In [22]:
rows_rdd.flatMap(lambda x:[x[:4] + [int(x[4])+1000] + x[5:]]).collect()

[['1001', 'C101', 'Bangalore', 'Laptop', 81000],
 ['1002', 'C102', 'Delhi', 'Phone', 31000],
 ['1003', 'C101', 'Bangalore', 'Mouse', 3000],
 ['1004', 'C103', 'Mumbai', 'Laptop', 76000],
 ['1005', 'C104', 'Delhi', 'Keyboard', 6000],
 ['1006', 'C101', 'Bangalore', 'Phone', 26000]]

In [23]:
rdd = sc.textFile('/root/pyspark/pyspark_practice_2026/Data/sample.txt')

In [25]:
rdd.union(rows_rdd).collect()

['A text file (sometimes spelled textfile; an old alternative name is flat file) is a kind of computer file that is structured as a sequence of lines of electronic text. A text file exists stored as data within a computer file system.',
 '',
 'In operating systems such as CP/M, where the operating system does not keep track of the file size in bytes, the end of a text file is denoted by placing one or more special characters, known as an end-of-file (EOF) marker, as padding after the last line in a text file.[1] In modern operating systems such as DOS, Microsoft Windows and Unix-like systems, text files do not contain any special EOF character, because file systems on those operating systems keep track of the file size in bytes.[2]',
 '',
 'Some operating systems, such as Multics, Unix-like systems, CP/M, DOS, the classic Mac OS, and Windows, store text files as a sequence of bytes, with an end-of-line delimiter at the end of each line. Other operating systems, such as OpenVMS and OS/3

In [26]:
sales = sc.parallelize([
    ("Bangalore", 80000),
    ("Delhi", 30000),
    ("Bangalore", 2000),
    ("Delhi", 5000),
    ("Mumbai", 75000)
])

In [30]:
sales.reduceByKey(lambda x, y: x+y).collect()

[('Delhi', 35000), ('Mumbai', 75000), ('Bangalore', 82000)]

In [32]:
sales.groupByKey().collect()

[('Delhi', <pyspark.resultiterable.ResultIterable at 0x74da82d17dd0>),
 ('Mumbai', <pyspark.resultiterable.ResultIterable at 0x74da8226f2c0>),
 ('Bangalore', <pyspark.resultiterable.ResultIterable at 0x74da8226d100>)]

In [33]:
sales.groupByKey().mapValues(list).collect()

[('Delhi', [30000, 5000]), ('Mumbai', [75000]), ('Bangalore', [80000, 2000])]

In [35]:
sales.groupByKey().mapValues(tuple).collect()

[('Delhi', (30000, 5000)), ('Mumbai', (75000,)), ('Bangalore', (80000, 2000))]

In [38]:
sales.groupBy(lambda x: x[0]).mapValues(list).collect()

[('Delhi', [('Delhi', 30000), ('Delhi', 5000)]),
 ('Mumbai', [('Mumbai', 75000)]),
 ('Bangalore', [('Bangalore', 80000), ('Bangalore', 2000)])]

In [39]:
orders = sc.parallelize([
    (1005, "Keyboard"),
    (1001, "Laptop"),
    (1003, "Mouse"),
    (1002, "Phone")
])

In [41]:
orders.sortByKey().collect()

[(1001, 'Laptop'), (1002, 'Phone'), (1003, 'Mouse'), (1005, 'Keyboard')]

In [42]:
customers = sc.parallelize([
    "C101",
    "C102",
    "C101",
    "C103",
    "C102"
])

In [43]:
customers.distinct().collect()

['C102', 'C103', 'C101']

In [44]:
customers = sc.parallelize([
    ("C101", "Basu"),
    ("C102", "Rahul"),
    ("C103", "Amit")
])

In [45]:
sales = sc.parallelize([
    ("C101", 80000),
    ("C102", 30000),
    ("C101", 2000)
])

In [46]:
sales.join(customers).collect()

[('C101', (80000, 'Basu')),
 ('C101', (2000, 'Basu')),
 ('C102', (30000, 'Rahul'))]

In [53]:
sales.getNumPartitions()

20

In [52]:
sales = sales.repartition(20)

In [55]:
sales.glom().collect()

[[],
 [],
 [],
 [],
 [],
 [],
 [('C101', 80000)],
 [],
 [],
 [],
 [('C102', 30000)],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [('C101', 2000)]]

In [57]:
sales = sales.coalesce(2)
sales.glom().collect()

[[('C101', 80000), ('C102', 30000), ('C101', 2000)]]

In [58]:
sales.count()

3

In [59]:
sales.first()

('C101', 80000)

In [61]:
sales.take(2)

[('C101', 80000), ('C102', 30000)]

In [62]:
sales.top(2)

[('C102', 30000), ('C101', 80000)]

In [64]:
amounts = sc.parallelize([100, 200, 300, 400])

In [65]:
total = amounts.reduce(
    lambda a, b: a + b
)

In [67]:
print(total)

1000


In [72]:
total = amounts.fold(
    0,
    lambda a, b: a + b
)

In [73]:
print(total)

1000


In [74]:
result = amounts.aggregate(
    (0, 0),
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda acc1, acc2: (
        acc1[0] + acc2[0],
        acc1[1] + acc2[1]
    )
)

In [77]:
print(result)

(1000, 4)


In [78]:
sales.foreach(lambda x:print(x))

('C101', 80000)
('C102', 30000)
('C101', 2000)


In [ ]:
sales.saveAsTextFile('')